# Synthetic Valley -- Particle Tracking

This example is based on the MODFLOW 6 example described [here](https://modflow6-examples.readthedocs.io/en/master/_examples/ex-gwt-synthetic-valley.html) and modified from the jupyter notebook [here](https://modflow6-examples.readthedocs.io/en/master/_notebooks/ex-gwt-synthetic-valley.html).  

The purpose of this notebook is to show how the MODFLOW 6 Particle Tracking (PRT) model can be used to simulate advective particle tracking paths.  The notebook concludes with a map that shows the ultimate destination of water table recharge.

In [ ]:
%matplotlib inline
import pathlib as pl
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import flopy
import synthetic_valley

silent = False

In [ ]:
ws = pl.Path("./temp") / "ex07c_sv"

## Create and run flow model

In [ ]:
sim_ws = ws / "flow"
sim = synthetic_valley.get_mf6gwf_sim(
    sim_ws, lake_concentration=1.0, recharge_concentration=0.0
)
sim.write_simulation(silent=silent)
sim.run_simulation(silent=silent)

In [ ]:
nper = sim.tdis.nper.array
gwf = sim.get_model("flow")
nlay, ncpl = gwf.disv.nlay.array, gwf.disv.ncpl.array
cell2d = gwf.disv.cell2d.array
vertices = gwf.disv.vertices.array
shape3d = (nlay, ncpl)

### Get the GWF grid cells connected to the lake

In [ ]:
pak = gwf.get_package("LAK-1")

In [ ]:
lak_conn = pak.connectiondata.array["cellid"]

In [ ]:
lak_conn

### Get the GWF grid cells connected to the stream

In [ ]:
pak = gwf.get_package("SFR_0")
sfr_conn = pak.packagedata.array["cellid"]

In [ ]:
sfr_conn

### Get the well locations

In [ ]:
pak = gwf.get_package("WEL_0")
well_loc = pak.stress_period_data.get_data(0)["cellid"]

In [ ]:
well_loc

### Create a zone array 

A different zone number will be specified for cells with lake, sfr, and well cells. This makes it easier to determine where particles terminate.


In [ ]:
izone = np.zeros(shape3d, dtype=int)
for cellid in lak_conn:
    izone[cellid] = 1
for cellid in sfr_conn:
    izone[cellid] = 2
for cellid in well_loc:
    izone[cellid] = 3
izone[0, 798 - 1] = 999

## Create a new PRT Model and Add to the Simulation

In [ ]:
# create the PRT model
prt_name = "sv_prt"
prt = flopy.mf6.ModflowPrt(
    sim,
    modelname=prt_name,
)

In [ ]:
# create PRT dis package
prt_dis = flopy.mf6.ModflowPrtdisv(
    prt,
    nlay=nlay,
    ncpl=ncpl,
    nvert=len(vertices),
    top=gwf.disv.top.array,
    botm=gwf.disv.botm.array,
    idomain=gwf.disv.idomain.array,
    cell2d=cell2d,
    vertices=vertices,
)

In [ ]:
# create PRT model input package
# set the porosity to gwf sy
mip = flopy.mf6.ModflowPrtmip(
    prt,
    pname="mip",
    porosity=0.2,
    izone=izone,
)

In [ ]:
# recharge release locations
top = gwf.disv.top.array
botm = gwf.disv.botm.array
xc, yc = (
    gwf.modelgrid.get_xcellcenters_for_layer(0),
    gwf.modelgrid.get_ycellcenters_for_layer(0),
)

xmin, xmax, ymin, ymax = gwf.modelgrid.extent
rch_release_loc = []
idx = 0
ifreq = 1
nodes = list(range(xc.flatten().shape[0]))[::ifreq]
xc = xc[::ifreq]
yc = yc[::ifreq]
for node, x, y in zip(nodes, xc, yc):
    # skip nodes with centroids on the edge of the model
    if np.allclose([x], [xmin]) or np.allclose([x], [xmax]):
        continue
    if np.allclose([y], [ymin]) or np.allclose([y], [ymax]):
        continue

    x, y = int(x), int(y)
    z = float(top[node])
    cellid = (0, node)

    # # skip lake cells
    # if cellid in lak_conn.tolist():
    #     continue
    if cellid in lak_conn.tolist():
        tag = "lake"
    else:
        tag = "land"

    rch_release_loc.append((idx, cellid, float(x), float(y), z, tag))
    idx += 1

len(rch_release_loc), rch_release_loc[:10]

In [ ]:
perlen = sim.tdis.perioddata.array["perlen"]
perlen / 365.

In [ ]:
dt = perlen[0] / (365.25 * 1.0)
dt = perlen[0] / 30.
nt = int(perlen[0] / dt)
tracktimes0 = [float(dt * idx) for idx in range(nt + 1)]

In [ ]:
trackcsvfile_record_rch = f"{prt_name}_rch.trk.csv"
prp_rch = flopy.mf6.ModflowPrtprp(
    prt,
    pname="prp_rch",
    filename=f"{prt_name}.rch.prp",
    trackcsv_filerecord=trackcsvfile_record_rch,
    drape=True,
    boundnames=True,
    dry_tracking_method="DROP",
    nreleasepts=len(rch_release_loc),
    packagedata=rch_release_loc,
    perioddata={0: ["first"]}, #, 2: []},
    exit_solve_tolerance=1e-5,
    extend_tracking=True,
    istopzone=999,
)

In [ ]:
budgetfile_record = f"{prt_name}.cbb"
prt_oc = flopy.mf6.ModflowPrtoc(
    prt,
    pname=f"{prt_name}-oc",
    budget_filerecord=budgetfile_record,
    ntracktimes=len(tracktimes0),
    tracktimes=[(t,) for t in tracktimes0],
    saverecord=[("BUDGET", "ALL")],
)

In [ ]:
ems = flopy.mf6.ModflowEms(
    sim,
    pname="ems",
    filename=f"{prt_name}.ems",
)
sim.register_solution_package(ems, [prt.name])

In [ ]:
exg = flopy.mf6.ModflowGwfprt(
    sim,
    exgtype="GWF6-PRT6",
    exgmnamea=gwf.name,
    exgmnameb=prt_name,
    filename=f"{gwf.name}.gwfprt",
)

## Write and run simulation

In [ ]:
sim.write_simulation()

In [ ]:
sim.run_simulation(silent=silent)

## Plot the results

### Forward particle tracks

In [ ]:
pathlines = pd.read_csv(ws / "flow" / trackcsvfile_record_rch)
pathlines

In [ ]:
term_pts = pathlines[pathlines.ireason == 1]
term_pts.t.describe()

In [ ]:
pathlines["izone"].unique(), pathlines["ilay"].unique()

In [ ]:
layer_colors = ["blue", "cyan", "green", "yellow", "red", "purple"]

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(3, 6), dpi=150)
    ax.set_aspect("equal")
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax)
    mm.plot_grid(lw=0.5, color="0.75")
    for k in range(0, nlay):
        df = pathlines[pathlines["ilay"] == k + 1]
        if len(df) > 0:
            mm.plot_pathline(df, layer="all", colors=layer_colors[k], linewidth=0.1)

In [ ]:
pathlines

In [ ]:
well_irpt = pathlines[(pathlines["istatus"] == 5) & (pathlines["izone"] == 3)][
    "irpt"
].unique()
df_well = pathlines[pathlines["irpt"].isin(well_irpt)]
df_well

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(3, 6), dpi=150)
    ax.set_aspect("equal")
    ax.set_title("Well Capture Zones")
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax)
    mm.plot_grid(lw=0.5, color="0.75")
    for k in range(nlay):
        df = df_well[df_well["ilay"] == k + 1]
        if len(df) > 0:
            mm.plot_pathline(df, layer="all", colors=layer_colors[k], linewidth=0.1)

In [ ]:
sfr_irpt = pathlines[(pathlines["istatus"] == 2) & (pathlines["izone"] == 2)][
    "irpt"
].unique()

sfr_irpt = pathlines[ (pathlines["izone"] == 2)][
    "irpt"
].unique()

df_sfr = pathlines[pathlines["irpt"].isin(sfr_irpt)]
sfr_irpt, df_sfr

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(3, 6), dpi=150)
    ax.set_aspect("equal")
    ax.set_title("Stream Capture Zones")
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax)
    mm.plot_grid(lw=0.5, color="0.75")
    for k in range(nlay):
        df = df_sfr[df_sfr["ilay"] == k + 1]
        if len(df) > 0:
            mm.plot_pathline(df, layer="all", colors=layer_colors[k], linewidth=0.1)

In [ ]:
lak_irpt = pathlines[(pathlines["istatus"] == 2) & (pathlines["izone"] == 1)][
    "irpt"
].unique()
lak_irpt = pathlines[(pathlines["izone"] == 1)][
    "irpt"
].unique()
df_lak = pathlines[pathlines["irpt"].isin(lak_irpt)]
df_lak

In [ ]:
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(3, 6), dpi=150)
    ax.set_title("Lake Capture Zones")
    ax.set_aspect("equal")
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax)
    mm.plot_grid(lw=0.5, color="0.75")
    for k in range(nlay):
        df = df_lak[df_lak["ilay"] == k + 1]
        if len(df) > 0:
            mm.plot_pathline(df, layer="all", colors=layer_colors[k], linewidth=0.1)

In [ ]:
irpt = pathlines[pathlines["izone"] == 1]["irpt"].unique()
df = pathlines[pathlines["irpt"].isin(irpt)]
ncpl = gwf.disv.ncpl.get_data()
iplot = np.zeros(ncpl, dtype=int)
iplot[df["icell"] - 1] = 1

fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(3, 6), dpi=150)
ax.set_title("Lake Capture")
ax.set_aspect("equal")
mm = flopy.plot.PlotMapView(model=gwf, ax=ax)
mm.plot_grid(lw=0.5, color="0.75")
mm.plot_array(iplot, cmap="viridis", vmin=0, vmax=1)
# ax.plot(df_lak["x"], df_lak["y"], "o", color="blue", markersize=0.5)


In [ ]:
irpt = pathlines[pathlines["izone"] == 2]["irpt"].unique()
df = pathlines[pathlines["irpt"].isin(irpt)]
df = df[df["ilay"] == 1]
ncpl = gwf.disv.ncpl.get_data()
iplot = np.zeros(ncpl, dtype=int)
iplot[df["icell"] - 1] = 1

fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(3, 6), dpi=150)
ax.set_title("Stream Capture")
ax.set_aspect("equal")
mm = flopy.plot.PlotMapView(model=gwf, ax=ax)
mm.plot_grid(lw=0.5, color="0.75")
mm.plot_array(iplot, cmap="viridis", vmin=0, vmax=1)

In [ ]:
irpt = pathlines[pathlines["izone"] == 3]["irpt"].unique()
df = pathlines[pathlines["irpt"].isin(irpt)]
df = df[df["ilay"] == 1]
ncpl = gwf.disv.ncpl.get_data()
iplot = np.zeros(ncpl, dtype=int)
iplot[df["icell"] - 1] = 1

fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(3, 6), dpi=150)
ax.set_title("Well Capture")
ax.set_aspect("equal")
mm = flopy.plot.PlotMapView(model=gwf, ax=ax)
mm.plot_grid(lw=0.5, color="0.75")
mm.plot_array(iplot, cmap="viridis", vmin=0, vmax=1)

In [ ]:
ncpl = gwf.disv.ncpl.get_data()
iplot = np.zeros(ncpl, dtype=int)

irpt = pathlines[pathlines["izone"] == 1]["irpt"].unique()
df = pathlines[pathlines["irpt"].isin(irpt)]
iplot[df["icell"] - 1] = 1

irpt = pathlines[pathlines["izone"] == 2]["irpt"].unique()
df = pathlines[pathlines["irpt"].isin(irpt)]
df = df[df["ilay"] == 1]
iplot[df["icell"] - 1] = 2

irpt = pathlines[pathlines["izone"] == 3]["irpt"].unique()
df = pathlines[pathlines["irpt"].isin(irpt)]
df = df[df["ilay"] == 1]
iplot[df["icell"] - 1] = 3

fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(3, 6), dpi=150)
ax.set_title("Capture")
ax.set_aspect("equal")
mm = flopy.plot.PlotMapView(model=gwf, ax=ax)
mm.plot_grid(lw=0.5, color="0.75")
img = mm.plot_array(iplot, cmap="viridis", vmin=0, vmax=3)
plt.colorbar(img, ax=ax, label="Destination Zone", shrink=0.5, ticks=[0.5, 1.5, 2.5], format="%d")